# DP-SGD on Criteo

We run classical DP-SGD end to end on the criteo dataset. We show each main piece of the dimma library: 
the loader, the accountant in both directions, the training loop, and the reference model.

In [ ]:
import time
from typing import NamedTuple

import jax
import matplotlib.pyplot as plt
import numpy as np

from dimma.accounting.sampling import (
    PoissonGaussianSchedule,
    calibrate_noise_multiplier,
    poisson_gaussian_epsilon,
)
from dimma.algorithms.dp_sgd import train as dp_sgd
from dimma.core import updates
from dimma.datasets.criteo import load_criteo
from dimma.metrics.calibration import (
    calibration_ratio,
    expected_calibration_error,
    reliability_curve,
)
from dimma.metrics.scoring import log_loss, normalized_entropy
from dimma.models.logreg import forward, init_params
from dimma.models.losses import per_sample_bce_loss

DEVICE = "gpu" if any(d.platform == "gpu" for d in jax.devices()) else "cpu"
print(f"jax devices: {jax.devices()}  ->  DEVICE = {DEVICE!r}")

# Figure palette. Blue carries the private run and every calibration number
# read off it, orange the noiseless control, aqua the operating-point panel.
# The three clear the colour-vision and normal-vision separation floors as a
# set; aqua sits under 3:1 against white, which is why every point it draws
# is also printed as a table above the figure.
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
PRIVATE_C, CONTROL_C = "#2a78d6", "#eb6834"
LOSS_C = ECE_C = PRIVATE_C
RECALL_C = "#1baf7a"

In [ ]:
TARGET_DELTA = 1e-6
TARGET_EPSILON = 3.0
STEPS = 10_000
EXPECTED_BATCH_SIZE = 1024

CLIP_NORMS = (3.5, 4.25, 5.0)
LEARNING_RATES = (0.01, 0.1, 0.5)

FEATURE_NORM_BOUND = None

## 1. The data

We will use criteo's 39 features: the 13 integer columns and the 26 categorical ones. The split
is deterministic given `seed` and `test_fraction`, so every run below sees
exactly the same rows.

In [3]:
split = load_criteo(features="all", preprocess=True, standardize=True,
                    download=False, seed=0,
                    device=DEVICE,
                    feature_norm_bound=FEATURE_NORM_BOUND
                    )

n_train, n_features = split.x_train.shape
base_rate = float(split.y_test.mean())
print(f"train {split.x_train.shape}   test {split.x_test.shape}")
print(f"label base rate: {float(split.y_train.mean()):.4f} train, "
      f"{base_rate:.4f} test")
print(f"PR-AUC floor (a random ranking scores the base rate): {base_rate:.4f}")

criteo: 39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. All 39 columns then standardized by the train-split mean and standard deviation.


train (800000, 39)   test (200000, 39)
label base rate: 0.2510 train, 0.2520 test
PR-AUC floor (a random ranking scores the base rate): 0.2520

enforced feature norm bound R = none, uncapped

39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. All 39 columns then standardized by the train-split mean and standard deviation.


### Relevant point to consider

Those preprocessing statistics (the medians, the means, the standard
deviations) are fitted on the training split and **accounted for in no
privacy budget**. The epsilon reported in this notebook
covers the training loop and nothing else.

## 2. Calibrating: a budget buys a noise multiplier

The accountant runs in both directions. This is the calibrating one: given a
target `(epsilon, delta)` and the shape of the run, it returns the smallest
noise multiplier that meets the budget.

`calibrate_noise_multiplier` takes a *builder* — a function from a candidate
multiplier to the release schedules a run would have at that multiplier —
because a method may release on more than one schedule. Plain DP-SGD has
exactly one, so the builder returns a single-entry list.

In [4]:
q = EXPECTED_BATCH_SIZE / n_train
print(f"sampling rate q = {q:.6f}, over {STEPS} steps")

t0 = time.time()
sigma = calibrate_noise_multiplier(
    lambda multiplier: [PoissonGaussianSchedule(q, multiplier, STEPS)],
    TARGET_EPSILON, TARGET_DELTA,
)
print(f"noise multiplier sigma = {sigma:.6f}   ({time.time() - t0:.1f}s)")

sampling rate q = 0.001280, over 10000 steps
noise multiplier sigma = 0.682662   (0.2s)


And the reporting direction, on the multiplier we just got back. It should
land at or just under the target — the budget is met, not approached from
above.

Note what the multiplier does **not** depend on: `C` does not appear anywhere
in this section. Clipping makes the sensitivity `C` and the noise scale
`sigma * C`, so the ratio the accountant sees is the same at every `C` in the
sweep. All nine runs in section 4 spend exactly this epsilon.

In [5]:
spent = poisson_gaussian_epsilon(q, sigma, STEPS, TARGET_DELTA)
print(f"epsilon at that multiplier = {spent:.6f}  (target {TARGET_EPSILON})")
assert spent <= TARGET_EPSILON + 1e-9

epsilon at that multiplier = 2.999999  (target 3.0)


## 3. Training, and where the threshold comes from

`train` takes a per-sample loss over arbitrary pytrees and never imports the
model, so the model and the algorithm meet only at this call.

Note what it does **not** return: no optimizer state, and no privacy cost. The
first is deliberate — accepting one would let a caller resume a run and replay
its noise stream from the start. The second is because the loop is not in a
position to claim an epsilon; that is the accountant's job, above.

**The threshold.** A confusion matrix needs an operating point, and `p > 0.5`
is the wrong one here. At a 25% base rate a calibrated model rarely emits
`p > 0.5` at all, so that threshold reports a few per cent recall for a model
whose ranking is fine — it measures the base rate, not the classifier. So the
threshold is the **training-split base rate**: flag a record when the model
puts it above average. It is fixed before any run, it is the same number for
every row of every table below, and it is a training-split statistic, so it
falls under the caveat already recorded above rather than adding a new read of
the test labels.

**The five numbers, and why each one.** All of them come from
`dimma.metrics`, which carries the reasoning; only the reason for reporting
it here is below.

- **Log-loss** — the objective the loop descends, so selection and training
  agree on what better means. Reported beside a ranking score because that
  pair is the CTR benchmark convention, where 0.001 absolute is the
  significance threshold (Zhu et al., *Open Benchmarking for CTR
  Prediction*, [arXiv:2009.05794](https://arxiv.org/abs/2009.05794)).
- **Normalized entropy** — log-loss over the entropy of the base rate, so
  1.0 is the constant predictor and the distance below it is the share of
  the available uncertainty removed (He et al., *Practical Lessons from
  Predicting Clicks on Ads at Facebook*, ADKDD 2014).
- **Calibration ratio** — observed clicks over predicted clicks. 1.0 is
  calibrated; a bid built on a model at 0.9 overpays by about 11% (He et
  al., same paper).
- **ECE** — the reliability curve's distance from the diagonal in one
  number, at 15 equal-mass bins. It is the metric this notebook gained
  because **per-example clipping, not the noise, is the major cause of
  miscalibration under DP** (Zhang et al., *A Closer Look at the
  Calibration of Differentially Private Learners*,
  [arXiv:2210.08248](https://arxiv.org/abs/2210.08248)) — which is why the
  sweep below is over `C`, and why the winner is chosen on log-loss rather
  than on the ranking. The estimator is biased upward and the bias grows
  with the bin count, so these compare only against each other.
- **PR-AUC** — the ranking read, floored at the base rate. It stays a local
  helper: `dimma.metrics` offers no metric that needs an operating point or
  reads rank alone, and `tests/metrics/test_package_surface.py` pins that
  absence.

Zhang et al. measure vision and language models, not tabular CTR. Carrying
their finding to Criteo is an extrapolation, and the reliability diagram
below is what checks it here.

In [ ]:
Y_TEST = np.asarray(split.y_test, dtype=np.float64)
THRESHOLD = float(split.y_train.mean())
THRESHOLD_LOGIT = float(np.log(THRESHOLD / (1.0 - THRESHOLD)))
N_BINS = 15
print(f"decision threshold p > {THRESHOLD:.4f}  "
      f"(logit > {THRESHOLD_LOGIT:.4f})")


class Eval(NamedTuple):
    """Everything one run is scored on, plus what drew it."""

    loss: float
    ne: float
    ece: float
    cal_ratio: float
    pr_auc: float
    confusion: tuple[int, int, int, int]
    probs: np.ndarray
    """Kept so the reliability curve does not cost a second training run."""


def average_precision(probs, labels):
    """Area under the precision-recall curve: sort descending, integrate.

    Local by decision rather than by omission -- see the note above.
    """
    desc = np.argsort(-probs)
    hits = labels[desc]
    precision = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return float((precision * hits).sum() / labels.sum())


def evaluate(params):
    """Score one parameter set on the test split.

    The metrics take probabilities, so the logits are squashed once here
    and every number below reads the same array.
    """
    logits = jax.vmap(forward, in_axes=(None, 0))(params, split.x_test)
    probs = np.asarray(jax.nn.sigmoid(logits), dtype=np.float64)

    pred = probs > THRESHOLD
    truth = Y_TEST > 0.5
    confusion = (int((~pred & ~truth).sum()), int((pred & ~truth).sum()),
                 int((~pred & truth).sum()), int((pred & truth).sum()))

    return Eval(
        loss=log_loss(probs, Y_TEST),
        ne=normalized_entropy(probs, Y_TEST),
        ece=expected_calibration_error(probs, Y_TEST, n_bins=N_BINS),
        cal_ratio=calibration_ratio(probs, Y_TEST),
        pr_auc=average_precision(probs, Y_TEST),
        confusion=confusion,
        probs=probs,
    )


def rates(confusion):
    """Precision and recall (TPR) from a confusion matrix."""
    tn, fp, fn, tp = confusion
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    return precision, recall


def show_confusion(confusion, title):
    """The 2x2 as a table -- four numbers do not need a chart."""
    tn, fp, fn, tp = confusion
    precision, recall = rates(confusion)
    print(f"{title}")
    print(f"                 predicted no    predicted click")
    print(f"  actual no      {tn:>12,}    {fp:>15,}")
    print(f"  actual click   {fn:>12,}    {tp:>15,}")
    print(f"  precision {precision:.4f}   recall {recall:.4f}   "
          f"(at threshold {THRESHOLD:.4f})")


def run(*, steps, clip_norm, learning_rate, noise_multiplier,
        expected_batch_size=EXPECTED_BATCH_SIZE, seed=0):
    """One complete DP-SGD run, from a fresh initialization."""
    params = init_params(jax.random.key(0), n_features)
    trained = dp_sgd.train(
        per_sample_bce_loss, params, updates.sgd(learning_rate),
        split.x_train, split.y_train,
        jax.random.key(seed), np.random.default_rng(seed),
        steps=steps, expected_batch_size=expected_batch_size,
        clip_norm=clip_norm, noise_multiplier=noise_multiplier,
    )
    return evaluate(trained)

The reference every log-loss below is read against: the best constant
predictor. It ignores the features entirely and always emits the training base
rate, so any model that scores worse than this one has learned something worse
than nothing — however good its ranking is.

Normalized entropy states the same anchor as 1.0, and the two disagree in the
fourth decimal because this constant is the *training* base rate while the
normalization divides by the entropy of the *test* one. Its PR-AUC is entered
as the floor rather than computed: every score is identical, so an average
precision over them would be reporting the order the rows arrived in.

In [ ]:
CONST_PROBS = np.full_like(Y_TEST, THRESHOLD)
CONST_LOSS = log_loss(CONST_PROBS, Y_TEST)
CONST = Eval(
    loss=CONST_LOSS,
    ne=normalized_entropy(CONST_PROBS, Y_TEST),
    ece=expected_calibration_error(CONST_PROBS, Y_TEST, n_bins=N_BINS),
    cal_ratio=calibration_ratio(CONST_PROBS, Y_TEST),
    pr_auc=base_rate,
    confusion=(0, int((Y_TEST < 0.5).sum()), 0, int((Y_TEST > 0.5).sum())),
    probs=CONST_PROBS,
)

start = init_params(jax.random.key(0), n_features)
before = evaluate(start)

COLUMNS = f"{'log-loss':>9} {'NE':>7} {'ECE':>7} {'cal-ratio':>10} {'PR-AUC':>7}"


def score_row(label, result, width=26):
    """One row of every summary table below, so they share a shape."""
    return (f"{label:<{width}}{result.loss:9.4f} {result.ne:7.4f} "
            f"{result.ece:7.4f} {result.cal_ratio:10.4f} {result.pr_auc:7.4f}")


print(f"{'':<26}{COLUMNS}")
print(score_row(f"constant at p = {THRESHOLD:.4f}", CONST))
print(score_row("at initialization", before))

## 4. The sweep: `C` against the learning rate

Nine runs, every pair of `C` in `CLIP_NORMS` and learning rate in
`LEARNING_RATES`, all at the same 10,000 steps and the same noise multiplier —
so **every row costs the same epsilon**, and the table is a utility comparison
at a fixed budget rather than a trade-off curve.

The bracket is where it is because `C` is only interesting near the gradient
norms it acts on. Far above them it clips nothing and does nothing; far below
it rescales every per-sample gradient to the same length, which replaces the
mean gradient with the mean of *normalized* gradients — a different update
direction, with no reason to converge to the logistic optimum.

In [ ]:
sweep = []
t0 = time.time()
print(f"{'C':>5} {'lr':>5}  {COLUMNS} {'prec':>7} {'recall':>7}")
for clip_norm in CLIP_NORMS:
    for learning_rate in LEARNING_RATES:
        result = run(steps=STEPS, clip_norm=clip_norm,
                     learning_rate=learning_rate, noise_multiplier=sigma)
        sweep.append((clip_norm, learning_rate, result))
        precision, recall = rates(result.confusion)
        print(f"{clip_norm:>5} {learning_rate:>5}  "
              f"{result.loss:9.4f} {result.ne:7.4f} {result.ece:7.4f} "
              f"{result.cal_ratio:10.4f} {result.pr_auc:7.4f} "
              f"{precision:7.4f} {recall:7.4f}")
print(f"\n{len(sweep)} runs in {time.time() - t0:.0f}s on {DEVICE}, "
      f"each at epsilon = {spent:.3f}, delta = {TARGET_DELTA}")

The best of the nine **by log-loss**, with its confusion matrix. Selection is
on the proper score rather than on the ranking because clipping is what damages
calibration, so the ranking is the half of the model this run is least likely
to have moved — PR-AUC is reported, not obeyed.

Beside it, the same configuration with the noise switched off
(`noise_multiplier = 1e-8`) — that is the control that isolates what *privacy*
cost, as distinct from what clipping and Poisson sampling cost.

That control is **not** a non-private baseline in the sense ADR-0005 means. It
still clips every per-sample gradient and still samples by Poisson. The real
baseline is a different loop, and it is notebook 3's dependency.

In [ ]:
BEST_C, BEST_LR, best = min(sweep, key=lambda row: row[2].loss)
print(f"best by log-loss: C = {BEST_C}, lr = {BEST_LR}\n")

ctrl = run(steps=STEPS, clip_norm=BEST_C, learning_rate=BEST_LR,
           noise_multiplier=1e-8)

print(f"{'':<26}{COLUMNS}")
print(score_row("constant / PR-AUC floor", CONST))
print(score_row("at initialization", before))
print(score_row(f"at epsilon = {spent:.3f}", best))
print(score_row("no noise (control)", ctrl))
print()
show_confusion(best.confusion, f"C = {BEST_C}, lr = {BEST_LR}, {STEPS:,} "
                               f"steps, epsilon = {spent:.3f}")
print()
show_confusion(ctrl.confusion, f"C = {BEST_C}, lr = {BEST_LR}, {STEPS:,} "
                               f"steps, no noise (control)")

### Reading the nine rows

**All nine beat the constant predictor**, which is the first thing to check and
not a given: a `C` far below the gradient norms produces a model that ranks
acceptably and is calibrated *worse than a model with no features at all*.
Inside this bracket that failure is behind us, and log-loss is doing real work.

**`C` is the axis that matters; the learning rate is nearly inert.** Read down
the log-loss column and `C` moves it monotonically — about 0.522 at 3.5, 0.516
at 4.25, 0.514 at 5.0 — at every one of the three learning rates. Read across
and a fiftyfold change in the learning rate, 0.01 to 0.5, moves it by one to
two thousandths — and in the *worse* direction.
That is worth stating plainly because it inverts the usual order of blame: the
learning rate is the knob one reaches for, and here it is not the one that is
binding.

**What `C` actually buys is recall, paid for in precision.** From `C = 3.5` to
`C = 5.0` recall goes 0.559 → 0.635 and precision 0.418 → 0.396. A larger `C`
leaves more of each per-sample gradient intact, the updates are less distorted,
and the model becomes readier to call a record a click. The ranking barely
moves; where the threshold falls in the score distribution does.

**Do not over-read the winner.** All nine PR-AUCs sit between 0.4434 and
0.4469. That is a spread of 0.0035, from one seed, with no repeats and so no
error bar. `C = 5.0, lr = 0.1` is the argmax and nothing stronger; the honest
claim is that the whole bracket performs about the same on ranking and that
`C` separates them on calibration.

**The bracket may not contain the optimum.** The best row sits at its top edge
and the log-loss trend is still descending at `C = 5.0`. This sweep says which
of these nine to use, not where the maximum is.

**And the privacy cost here is almost exactly nothing.** The noiseless control
scores PR-AUC 0.4471 against the private run's 0.4469, at the same log-loss to
four decimals. At `expected_batch_size = 1024` the calibrated multiplier adds
Gaussian noise of scale `sigma * C` to a sum of about a thousand clipped
gradients, so the perturbation is small next to the signal it is hiding in.
On this problem, at this budget and this batch size, **what costs utility is
the clipping, not the noise** — which is why the sweep above is over `C` and
why section 5 does not find more steps buying anything.

### Where the probabilities actually land

ECE collapses this figure to one number; this is the figure. Each point is a
bin of the test split: what the model said on the horizontal, what happened on
the vertical. Points below the diagonal are over-prediction, above it
under-prediction, and the private run and the noiseless control are drawn
together because the difference between them is the whole question — if
clipping rather than noise is what miscalibrates, the two curves leave the
diagonal together rather than apart.

Bins are equal-mass, so each holds about the same number of records and no
point is estimated from far fewer than its neighbours.

In [ ]:
best_curve = reliability_curve(best.probs, Y_TEST, n_bins=N_BINS)
ctrl_curve = reliability_curve(ctrl.probs, Y_TEST, n_bins=N_BINS)

points = np.concatenate([best_curve.mean_predicted, best_curve.mean_observed,
                         ctrl_curve.mean_predicted, ctrl_curve.mean_observed])
pad = 0.05 * (points.max() - points.min())
lo, hi = points.min() - pad, points.max() + pad

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1, color=MUTED,
        zorder=1, label="perfectly calibrated")

# Identity is in the legend rather than at the end of each line: the two
# curves are expected to sit almost on top of each other and on the
# diagonal, which is where a direct label stops being readable.
for reliability, colour, label in (
    (best_curve, PRIVATE_C, f"epsilon = {spent:.3f}    ECE {best.ece:.4f}"),
    (ctrl_curve, CONTROL_C, f"no noise (control)    ECE {ctrl.ece:.4f}"),
):
    ax.plot(reliability.mean_predicted, reliability.mean_observed, marker="o",
            markersize=7, linewidth=2, color=colour, label=label,
            markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=2)

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed click rate")
ax.set_title(f"Reliability at C = {BEST_C}, lr = {BEST_LR},\n"
             f"{N_BINS} equal-mass bins", color=INK)
ax.grid(alpha=0.5)
ax.legend(loc="upper left", frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

print(f"private: {len(best_curve.count)} bins kept, "
      f"{best_curve.count.min():,}-{best_curve.count.max():,} records each, "
      f"largest gap {np.abs(best_curve.gap).max():.4f}")
print(f"control: {len(ctrl_curve.count)} bins kept, "
      f"{ctrl_curve.count.min():,}-{ctrl_curve.count.max():,} records each, "
      f"largest gap {np.abs(ctrl_curve.gap).max():.4f}")

## 5. The best configuration over training steps

Hold `C`, the learning rate and the noise multiplier at the values section 4
picked, and vary only how long the run trains. Log-loss, ECE, and recall —
three quantities on three scales, so three panels, never two axes on one plot.
The first two are the proper score and the calibration gap moving separately;
the third is what that does at the one operating point this notebook fixes.

**Each point is an independent run, not a trajectory.** That is forced rather
than chosen: `train` returns no optimizer state precisely so that a run cannot
be resumed, because resuming would replay the noise stream. So the curve is
assembled from separate runs, each starting from the same initialization and
each drawing its own noise.

The noise multiplier is held at the value calibrated for 10,000 steps, so the
shorter runs are *over*-noised for their length and spend correspondingly less
epsilon. The epsilon each point actually spends is in the table.

In [ ]:
step_grid = [50, 100, 200, 500, 1000, 2000, 4000, 7000, 10_000]
curve = []
print(f"{'steps':>7} {'epsilon':>8}  {COLUMNS} {'prec':>7} {'recall':>7}")
for steps in step_grid:
    result = run(steps=steps, clip_norm=BEST_C, learning_rate=BEST_LR,
                 noise_multiplier=sigma)
    eps = poisson_gaussian_epsilon(q, sigma, steps, TARGET_DELTA)
    precision, recall = rates(result.confusion)
    curve.append((steps, eps, result))
    print(f"{steps:>7,} {eps:>8.3f}  "
          f"{result.loss:9.4f} {result.ne:7.4f} {result.ece:7.4f} "
          f"{result.cal_ratio:10.4f} {result.pr_auc:7.4f} "
          f"{precision:7.4f} {recall:7.4f}")

In [ ]:
steps_axis = [point[0] for point in curve]
results = [point[2] for point in curve]

fig, (left, middle, right) = plt.subplots(1, 3, figsize=(15, 4))

left.plot(steps_axis, [r.loss for r in results], marker="o", markersize=7,
          linewidth=2, color=LOSS_C, markeredgecolor=SURFACE,
          markeredgewidth=1.5)
left.axhline(CONST_LOSS, color=MUTED, linestyle=":", alpha=0.9)
left.annotate("constant predictor", xy=(step_grid[0], CONST_LOSS),
              xytext=(0, 5), textcoords="offset points", color=MUTED,
              fontsize=9)
left.set_ylabel("test log-loss")
left.set_title("Score")

# ECE is an upward-biased estimate, so the distance from zero is not a
# calibration error and only the movement across this panel is readable --
# every point is the same n_bins on the same evaluation set.
middle.plot(steps_axis, [r.ece for r in results], marker="o", markersize=7,
            linewidth=2, color=ECE_C, markeredgecolor=SURFACE,
            markeredgewidth=1.5)
middle.set_ylabel(f"test ECE, {N_BINS} equal-mass bins")
middle.set_title("Calibration")
middle.set_ylim(bottom=0.0)

right.plot(steps_axis, [rates(r.confusion)[1] for r in results], marker="o",
           markersize=7, linewidth=2, color=RECALL_C,
           markeredgecolor=SURFACE, markeredgewidth=1.5)
right.set_ylabel(f"test recall at p > {THRESHOLD:.3f}")
right.set_title("Clicks caught")
# A rate gets its own scale. Autoscaling this panel would fill the frame with
# the seed-to-seed wobble in the converged tail and draw it as a trend.
right.set_ylim(0.0, 1.02)

for axis in (left, middle, right):
    axis.set_xlabel("training steps")
    axis.set_xscale("log")
    axis.grid(alpha=0.5)

fig.suptitle(f"DP-SGD on Criteo at C = {BEST_C}, lr = {BEST_LR}, "
             f"noise multiplier {sigma:.3f}", color=INK)
fig.tight_layout()
plt.show()

### Reading those three panels

**The run is finished by about 200 steps.** Log-loss falls 0.5361 → 0.5149 over
the first 200 and then moves in the fourth decimal for the remaining 9,800.
Recall does the same thing on the same timescale. Note the x-axis is
logarithmic: on a linear axis all of this happens inside the first pixel, which
is exactly how the earlier version of this figure managed to show a flat line
and call it convergence.

**Recall falls, and that is the model improving, not degrading.** At 50 steps
the parameters have barely left their initialization, nearly every record
scores above the base-rate threshold, and the model catches 88% of the clicks
at 31% precision — it is close to flagging everyone, which is what a model that
knows nothing does at this threshold. By 200 steps it catches 65% at 39%. The
count of clicks caught goes down because the model has stopped calling
everything a click; the precision column in the table above is the other half
of that sentence, and log-loss on the left panel confirms it independently.

**The wobble after 500 steps is one seed, not a trend.** Recall wanders between
0.624 and 0.638 and log-loss between 0.5140 and 0.5147, with no ordering. Each
point is an independent run with its own noise draw, so that band is what a
single seed's variation looks like — which is why the recall axis is pinned to
the full 0–1 range it lives on. Autoscaled, that same band fills the panel and
reads as structure.

**The steps after the first few hundred cost privacy and buy nothing.** From
200 steps to 10,000, epsilon climbs 2.182 → 3.000 and log-loss improves by
0.0008. On this problem, at this batch size, a longer run is not how the budget
should be spent.

Two things that stop that from being a general claim. The noise multiplier is
held at the value calibrated for 10,000 steps, so every shorter run here is
*over*-noised for its length: a run actually targeting epsilon = 2.075 would
calibrate a smaller multiplier and land above its point on these curves. And
PR-AUC, which these panels do not show, is still creeping upward late
(0.4311 at 50 steps, 0.4480 at 7,000) — the ranking keeps sharpening a little
after the calibration has settled. The table above carries both.

## What this shows, and what it does not

Working, as far as this goes: the loader, the accountant in both directions
(calibrate then report, agreeing to floating-point), the DP-SGD loop on 800k
real rows and all 39 features, and the reference model.

Not shown, and not claimed:

- **No comparison.** One algorithm, no baseline. Notebooks 2 and 3.
- **The preprocessing statistics are unaccounted** (ADR-0008). The epsilon
  above covers the training loop only.
- **There was a hyperparameter search, and it is unaccounted twice over.**
  Section 4 trains nine configurations and keeps one. `accounting/sampling.py`
  lists hyperparameter search among the things it never accounts for, so the
  epsilon beside the winning row is the epsilon of *that run*, not of the
  procedure that found it. And the winner was picked by a metric computed on
  the test split, which makes the reported figure optimistic as an estimate of
  held-out performance as well as unpaid for as a privacy matter. Both are
  fine for a smoke test that claims nothing; neither is fine once a number is
  reported as a guarantee.
- **The operating point is a choice.** Every confusion matrix thresholds at the
  training base rate. A different threshold gives different precision and
  recall from the very same model — the ranking, which PR-AUC reads, is what
  does not move.
- **No `R`.** This notebook caps no feature norms (section 1), which is
  available to it only because clipping is what bounds DP-SGD's sensitivity.
  Notebook 2 has to pick one, apply it to both arms, and say where it came
  from (ADR-0012).
- **One seed.** Every run above is `seed=0`, so none of the gaps come with an
  error bar. Distinguishing two methods needs repeats; that is notebook 2's
  problem, not this one's.
- **The epsilon is RDP's** (ADR-0011). PLD would report a smaller number for
  the same run; whichever is used has to be held fixed across a comparison and
  reported alongside it.